In [3]:
# unzip file
from zipfile import ZipFile

fileName = 'archive'

with ZipFile("demo_data/"+fileName+".zip", 'r') as zObject:
    zObject.extractall(
      path="demo_data/")

In [101]:
"""
Use patchify....
Tile 1: 797 x 644 --> 768 x 512 --> 6
Tile 2: 509 x 544 --> 512 x 256 --> 2
Tile 3: 682 x 658 --> 512 x 512  --> 4
Tile 4: 1099 x 846 --> 1024 x 768 --> 12
Tile 5: 1126 x 1058 --> 1024 x 1024 --> 16
Tile 6: 859 x 838 --> 768 x 768 --> 9
Tile 7: 1817 x 2061 --> 1792 x 2048 --> 56
Tile 8: 2149 x 1479 --> 1280 x 2048 --> 40
Total 9 images in each folder * (145 patches) = 1305
Total 1305 patches of size 256x256
"""

import os
import cv2
import numpy as np

from matplotlib import pyplot as plt
from patchify import patchify
from PIL import Image
import segmentation_models as sm
from tensorflow.keras.metrics import MeanIoU

from sklearn.preprocessing import MinMaxScaler, StandardScaler
scaler = MinMaxScaler()

#root_directory = 'demo_data/Semantic segmentation dataset/'
root_directory = 'demo_data/sentinel/'

patch_size = 256

#Read images from repsective 'images' subdirectory
#As all images are of ddifferent size we have 2 options, either resize or crop
#But, some images are too large and some small. Resizing will change the size of real objects.
#Therefore, we will crop them to a nearest size divisible by 256 and then 
#divide all images into patches of 256x256x3. 
image_dataset = []  
for path, subdirs, files in os.walk(root_directory):
    #print(path)  
    dirname = path.split(os.path.sep)[-1]
    if dirname == 'images':   #Find all 'images' directories
        images = os.listdir(path)  #List of all image names in this subdirectory
        for i, image_name in enumerate(images):  
            if image_name.endswith(".jp2"):   #Only read jpg images...
               
                image = cv2.imread(path+"/"+image_name, 1)  #Read each image as BGR
                SIZE_X = (image.shape[1]//patch_size)*patch_size #Nearest size divisible by our patch size
                SIZE_Y = (image.shape[0]//patch_size)*patch_size #Nearest size divisible by our patch size
                image = Image.fromarray(image)
                image = image.crop((0 ,0, SIZE_X, SIZE_Y))  #Crop from top left corner
                #image = image.resize((SIZE_X, SIZE_Y))  #Try not to resize for semantic segmentation
                image = np.array(image)             
       
                #Extract patches from each image
                print("Now patchifying image:", path+"/"+image_name)
                patches_img = patchify(image, (patch_size, patch_size, 3), step=patch_size)  #Step=256 for 256 patches means no overlap
        
                for i in range(patches_img.shape[0]):
                    for j in range(patches_img.shape[1]):
                        
                        single_patch_img = patches_img[i,j,:,:]
                        
                        #Use minmaxscaler instead of just dividing by 255. 
                        single_patch_img = scaler.fit_transform(single_patch_img.reshape(-1, single_patch_img.shape[-1])).reshape(single_patch_img.shape)
                        
                        #single_patch_img = (single_patch_img.astype('float32')) / 255. 
                        single_patch_img = single_patch_img[0] #Drop the extra unecessary dimension that patchify adds.                               
                        image_dataset.append(single_patch_img)
                

/opt/conda/lib/python3.10/site-packages/PIL/Image.py:3167: DecompressionBombWarning: Image size (115605504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Now patchifying image: demo_data/sentinel/images/T37NCC_20230513T073619_TCI_10m.jp2


In [102]:
#image_dataset

[array([[[0.18823529, 0.15294118, 0.17647059],
         [0.15686275, 0.16470588, 0.2       ],
         [0.15686275, 0.15686275, 0.18039216],
         ...,
         [0.57254902, 0.81960784, 1.        ],
         [0.64313725, 0.84705882, 1.        ],
         [0.63921569, 0.84313725, 1.        ]],
 
        [[0.27843137, 0.2627451 , 0.27843137],
         [0.25098039, 0.27058824, 0.31764706],
         [0.23137255, 0.2627451 , 0.32156863],
         ...,
         [0.58431373, 0.80392157, 0.99607843],
         [0.65098039, 0.8627451 , 1.        ],
         [0.62745098, 0.84313725, 1.        ]],
 
        [[0.27843137, 0.27843137, 0.3254902 ],
         [0.2745098 , 0.30588235, 0.37647059],
         [0.2627451 , 0.32156863, 0.39215686],
         ...,
         [0.57254902, 0.78823529, 1.        ],
         [0.65490196, 0.8745098 , 0.99607843],
         [0.62352941, 0.84705882, 0.99607843]],
 
        ...,
 
        [[0.46666667, 0.59215686, 0.78431373],
         [0.49411765, 0.65098039, 0.82352

In [94]:
 
 #Now do the same as above for masks
 #For this specific dataset we could have added masks to the above code as masks have extension png
mask_dataset = []  
for path, subdirs, files in os.walk(root_directory):
    #print(path)  
    dirname = path.split(os.path.sep)[-1]
    if dirname == 'masks':   #Find all 'images' directories
        masks = os.listdir(path)  #List of all image names in this subdirectory
        for i, mask_name in enumerate(masks):  
            if mask_name.endswith(".png"):   #Only read png images... (masks in this dataset)
               
                mask = cv2.imread(path+"/"+mask_name, 1)  #Read each image as Grey (or color but remember to map each color to an integer)
                mask = cv2.cvtColor(mask,cv2.COLOR_BGR2RGB)
                SIZE_X = (mask.shape[1]//patch_size)*patch_size #Nearest size divisible by our patch size
                SIZE_Y = (mask.shape[0]//patch_size)*patch_size #Nearest size divisible by our patch size
                mask = Image.fromarray(mask)
                mask = mask.crop((0 ,0, SIZE_X, SIZE_Y))  #Crop from top left corner
                #mask = mask.resize((SIZE_X, SIZE_Y))  #Try not to resize for semantic segmentation
                mask = np.array(mask)             
       
                #Extract patches from each image
                print("Now patchifying mask:", path+"/"+mask_name)
                patches_mask = patchify(mask, (patch_size, patch_size, 3), step=patch_size)  #Step=256 for 256 patches means no overlap
        
                for i in range(patches_mask.shape[0]):
                    for j in range(patches_mask.shape[1]):
                        
                        single_patch_mask = patches_mask[i,j,:,:]
                        #single_patch_img = (single_patch_img.astype('float32')) / 255. #No need to scale masks, but you can do it if you want
                        single_patch_mask = single_patch_mask[0] #Drop the extra unecessary dimension that patchify adds.                               
                        mask_dataset.append(single_patch_mask) 
 

Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_002.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_005.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_006.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_007.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_008.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_003.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_004.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_001.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 8/masks/image_part_009.png
Now patchifying mask: demo_data/Semantic segmentation dataset/Tile 7/masks/image_part_002.png
Now patchifying mask: demo_data/Semantic segmentation datase

In [95]:
image_dataset = np.array(image_dataset)
mask_dataset =  np.array(mask_dataset)

In [96]:
#image_dataset


array([[[[0.63529412, 0.6       , 0.58167331],
         [0.61176471, 0.57647059, 0.55776892],
         [0.61568627, 0.58039216, 0.56175299],
         ...,
         [0.91764706, 0.90196078, 0.92031873],
         [0.85490196, 0.83921569, 0.85657371],
         [0.79215686, 0.77647059, 0.79282869]],

        [[0.64313725, 0.60784314, 0.58964143],
         [0.63137255, 0.59607843, 0.57768924],
         [0.62745098, 0.59215686, 0.57370518],
         ...,
         [0.9254902 , 0.90980392, 0.92828685],
         [0.83529412, 0.81960784, 0.83665339],
         [0.75686275, 0.74117647, 0.75697211]],

        [[0.63529412, 0.6       , 0.58167331],
         [0.63529412, 0.6       , 0.58167331],
         [0.62745098, 0.59215686, 0.57370518],
         ...,
         [0.85882353, 0.84705882, 0.85258964],
         [0.85490196, 0.84313725, 0.84860558],
         [0.86666667, 0.85490196, 0.86055777]],

        ...,

        [[0.98823529, 0.97647059, 0.96015936],
         [0.98039216, 0.96862745, 0.95219124]